In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("cleaned_weather_traffic_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166105 entries, 0 to 166104
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   key                   166105 non-null  object 
 1   fare_amount           166105 non-null  float64
 2   pickup_datetime       166105 non-null  object 
 3   pickup_longitude      166105 non-null  float64
 4   pickup_latitude       166105 non-null  float64
 5   dropoff_longitude     166105 non-null  float64
 6   dropoff_latitude      166105 non-null  float64
 7   passenger_count       166105 non-null  int64  
 8   pickup_hour           166105 non-null  object 
 9   temperature           166105 non-null  float64
 10  precipitation         166105 non-null  float64
 11  wind_speed            166105 non-null  float64
 12  distance_km           166105 non-null  float64
 13  hour                  166105 non-null  int64  
 14  is_peak               166105 non-null  bool   
 15  

In [3]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

In [4]:
df['date'] = df['pickup_datetime'].dt.date

In [5]:
df['day_name'] = df['pickup_datetime'].dt.day_name()

In [6]:
df['month'] = df['pickup_datetime'].dt.month
df['month_name'] = df['pickup_datetime'].dt.month_name()

In [7]:
df['is_weekend'] = df['day_name'].isin(['Saturday','Sunday'])

In [8]:
def time_bucket(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 22:
        return "Evening"
    else:
        return "Night"

df['time_bucket'] = df['hour'].apply(time_bucket)

In [9]:
df['revenue_per_km'] = df['fare_amount'] / df['distance_km']

In [10]:
df['revenue_per_min'] = df['fare_amount'] / df['duration_min']

In [11]:
df['is_high_fare'] = df['fare_amount'] > df['fare_amount'].quantile(0.75)

In [12]:
df['delay_ratio'] = df['traffic_delay'] / df['duration_min']

In [13]:
df['congestion_level'] = pd.cut(
    df['delay_ratio'],
    bins=[-1, 0.05, 0.15, 1],
    labels=['Low','Medium','High']
)

In [14]:
df['speed_category'] = pd.cut(
    df['avg_speed'],
    bins=[0, 20, 40, 200],
    labels=['Slow','Moderate','Fast']
)

In [15]:
df['rain_flag'] = df['precipitation'] > 0

In [16]:
df['rain_category'] = pd.cut(
    df['precipitation'],
    bins=[-1, 0, 2, 10, 100],
    labels=['No Rain','Light Rain','Moderate Rain','Heavy Rain']
)

In [17]:
df['temp_category'] = pd.cut(
    df['temperature'],
    bins=[-20, 5, 15, 25, 40],
    labels=['Cold','Cool','Warm','Hot']
)

In [18]:
df['trip_efficiency'] = df['distance_km'] / df['duration_min']

In [19]:
df['peak_rain'] = df['is_peak'] & df['rain_flag']

In [20]:
df['peak_congestion'] = df['is_peak'] & (df['congestion_level'] == 'High')

In [21]:
df['pickup_lat_cluster'] = df['pickup_latitude'].round(4)
df['pickup_lon_cluster'] = df['pickup_longitude'].round(4)

In [22]:
df.to_csv("final_engineered_dataset.csv", index=False)